# FE BAR Step 4 - Fraud batch scoring (Layer 3)

Scores every row of `workshop_guy_catalog.fe_bar_a0.txn_features_gold` with `models:/workshop_guy_catalog.fe_bar_a0.fraud_risk_model@champion` and overwrites `workshop_guy_catalog.fe_bar_a0.scored_transactions`.

* Thresholds and cohort cutoff are read from the champion run's MLflow params (nothing hardcoded).
* Per-row TreeSHAP; top 3 features by |SHAP| stored as JSON in `top_shap_signals`.
* Runs as task `batch_scoring` in job `fe_bar_gold_to_lakebase` on serverless.

In [0]:
%pip install -q lightgbm==4.7.0
dbutils.library.restartPython()

In [0]:
import json
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F

CATALOG, SCHEMA = "workshop_guy_catalog", "fe_bar_a0"
FEATURE_TABLE = f"{CATALOG}.{SCHEMA}.txn_features_gold"
SCORED_TABLE = f"{CATALOG}.{SCHEMA}.scored_transactions"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.fraud_risk_model"
MODEL_URI = f"models:/{MODEL_NAME}@champion"
mlflow.set_registry_uri("databricks-uc")
print("Model URI:", MODEL_URI, "| target:", SCORED_TABLE)

In [0]:
pre = spark.sql(f"""
  SELECT current_user() AS current_user,
         is_account_group_member('pii') AS is_pii_member,
         (SELECT count(*) FROM {FEATURE_TABLE}) AS total_rows,
         (SELECT count_if(amount > 10000) FROM {FEATURE_TABLE}) AS amount_gt_10000_rows,
         (SELECT count(amt_to_monthly_spend) FROM {FEATURE_TABLE}) AS non_null_amt_to_monthly_spend
""").collect()[0].asDict()
for k, v in pre.items():
    print(f"{k:32s} {v}")
if not pre["is_pii_member"] or pre["amount_gt_10000_rows"] == 0:
    raise RuntimeError("PRE-FLIGHT FAILED: not a pii member or amount > 10000 rows hidden. Scoring would run on "
                       "filtered/masked data. Stopping - flag to the owner.")
print("\nPRE-FLIGHT PASSED")
GOLD_ROWS = pre["total_rows"]

In [0]:
client = MlflowClient(registry_uri="databricks-uc")
mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
MODEL_VERSION = str(mv.version)
run_params = client.get_run(mv.run_id).data.params
BLOCK_T = float(run_params["block_threshold"])
REVIEW_T = float(run_params["review_threshold"])
COHORT_P90 = float(run_params["cohort_amt_ratio_p90"])

model = mlflow.pyfunc.load_model(MODEL_URI)
inner = model.unwrap_python_model()
FEATURES, CATEGORICAL, CAT_LEVELS = inner.features, inner.categorical, inner.cat_levels
BOOSTER_STR = inner.booster.model_to_string()

print(f"champion -> version {MODEL_VERSION}, run_id {mv.run_id}")
print(f"block_threshold={BLOCK_T:.6f} review_threshold={REVIEW_T:.6f} cohort_amt_ratio_p90={COHORT_P90:.4f} (from run params)")
print("features:", FEATURES)

In [0]:
NUMERIC = [f for f in FEATURES if f not in CATEGORICAL]
PASS_COLS = ["txn_id", "customer_id", "txn_ts", "amount"]
OUT_SCHEMA = "txn_id string, customer_id string, txn_ts timestamp, amount double, risk_score double, top_shap_signals string"

def _json_val(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    return round(float(v), 4) if isinstance(v, (int, float, np.integer, np.floating)) else str(v)

def score_partition(batches):
    import lightgbm as lgb
    booster = lgb.Booster(model_str=BOOSTER_STR)
    for pdf in batches:
        X = pdf[FEATURES].copy()
        for c in CATEGORICAL:
            X[c] = pd.Categorical(X[c].astype("object"), categories=CAT_LEVELS[c])
        for c in NUMERIC:
            X[c] = pd.to_numeric(X[c], errors="coerce").astype("float64")
        score = booster.predict(X)
        contrib = booster.predict(X, pred_contrib=True)[:, :-1]
        top = np.argsort(-np.abs(contrib), axis=1)[:, :3]
        raw = pdf[FEATURES].to_numpy(dtype=object)
        signals = [json.dumps([{"feature": FEATURES[j], "value": _json_val(raw[i, j]), "shap": round(float(contrib[i, j]), 4)}
                               for j in top[i]]) for i in range(len(pdf))]
        out = pdf[PASS_COLS].copy()
        out["risk_score"] = score
        out["top_shap_signals"] = signals
        yield out

scored = (spark.table(FEATURE_TABLE)
          .select(*PASS_COLS, *[c for c in FEATURES if c not in PASS_COLS])
          .repartition(64)
          .mapInPandas(score_partition, OUT_SCHEMA)
          .withColumn("recommended_action",
                      F.when(F.col("risk_score") >= BLOCK_T, "block")
                       .when(F.col("risk_score") >= REVIEW_T, "review")
                       .otherwise("approve"))
          .withColumn("model_version", F.lit(MODEL_VERSION))
          .withColumn("scored_at", F.current_timestamp())
          .select("txn_id", "customer_id", "txn_ts", "amount", "risk_score", "recommended_action",
                  "top_shap_signals", "model_version", "scored_at"))

# Consistency check: distributed booster scores == registered pyfunc predict() on a sample
sample = spark.table(FEATURE_TABLE).orderBy("txn_id").limit(1000).toPandas()
model_input = sample[FEATURES].copy()
for c in NUMERIC:
    model_input[c] = model_input[c].astype("float64")  # signature declares double inputs
pyfunc_scores = np.asarray(model.predict(model_input))
dist_scores = (scored.join(spark.createDataFrame(sample[["txn_id"]]), "txn_id").select("txn_id", "risk_score")
               .toPandas().set_index("txn_id").loc[sample.txn_id, "risk_score"].values)
print("max |pyfunc - distributed| on 1000 rows:", float(np.abs(pyfunc_scores - dist_scores).max()))

In [0]:
null_ids = spark.table(FEATURE_TABLE).filter("txn_id IS NULL").count()
dup_ids = spark.table(FEATURE_TABLE).groupBy("txn_id").count().filter("count > 1").count()
assert null_ids == 0 and dup_ids == 0, f"txn_id not a valid key: nulls={null_ids}, dups={dup_ids}"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SCORED_TABLE} (
  txn_id STRING NOT NULL,
  customer_id STRING,
  txn_ts TIMESTAMP,
  amount DOUBLE,
  risk_score DOUBLE,
  recommended_action STRING,
  top_shap_signals STRING COMMENT 'JSON array of top 3 feature/value/shap objects by abs SHAP (log-odds)',
  model_version STRING,
  scored_at TIMESTAMP,
  CONSTRAINT scored_transactions_pk PRIMARY KEY (txn_id)
) USING DELTA
COMMENT 'FE BAR Layer 3: batch fraud risk scores for every txn in txn_features_gold (overwritten each run)'
""")

scored.createOrReplaceTempView("scored_v")
spark.sql(f"INSERT OVERWRITE TABLE {SCORED_TABLE} SELECT * FROM scored_v")
h = spark.sql(f"DESCRIBE HISTORY {SCORED_TABLE} LIMIT 1").collect()[0]
print("Delta version:", h.version, "| operation:", h.operation, "| timestamp:", h.timestamp, "| metrics:", h.operationMetrics)

In [0]:
st = spark.table(SCORED_TABLE)
n = st.count()
print(f"scored_transactions rows: {n:,} | txn_features_gold rows: {GOLD_ROWS:,} | match: {n == GOLD_ROWS}")
assert n == GOLD_ROWS
print("\nConstraints:", [r.asDict() for r in spark.sql(f"""
  SELECT constraint_name, constraint_type FROM {CATALOG}.information_schema.table_constraints
  WHERE table_schema = '{SCHEMA}' AND table_name = 'scored_transactions'""").collect()])

print("\nrecommended_action distribution:")
print(st.groupBy("recommended_action").agg(F.count("*").alias("txns"))
        .withColumn("pct", F.round(F.col("txns") / n * 100, 3)).orderBy("txns").toPandas().to_string(index=False))

print("\nrisk_score percentiles:")
pcts = [0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
q = st.select(F.percentile_approx("risk_score", pcts, 10000).alias("q"), F.min("risk_score"), F.max("risk_score")).collect()[0]
print(pd.DataFrame({"percentile": [f"p{p*100:g}" for p in pcts], "risk_score": q[0]}).to_string(index=False))
print(f"min={q[1]:.6f} max={q[2]:.6f}")
print("\nmodel_version / scored_at:", st.select("model_version", "scored_at").distinct().collect())

pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 250)
print("\nTop 10 highest-risk transactions:")
print(st.orderBy(F.desc("risk_score"), "txn_id").limit(10)
        .select("txn_id", "customer_id", "txn_ts", "amount", F.round("risk_score", 6).alias("risk_score"),
                "recommended_action", "top_shap_signals").toPandas().to_string(index=False))

In [0]:
# Deterministic pick: seeded cohort (amt ratio >= train p90, cross-border, new device), action=block, confirmed fraud,
# highest risk_score (tie-break txn_id). Downstream validation uses this same txn_id.
trace = spark.sql(f"""
  SELECT s.txn_id, s.customer_id, s.txn_ts, s.amount, g.amt_to_monthly_spend, g.is_cross_border, g.is_new_device,
         g.is_fraud, s.risk_score, s.recommended_action, s.top_shap_signals, s.model_version, s.scored_at
  FROM {SCORED_TABLE} s JOIN {FEATURE_TABLE} g USING (txn_id)
  WHERE g.amt_to_monthly_spend >= {COHORT_P90} AND g.is_cross_border = 1 AND g.is_new_device = 1
    AND s.recommended_action = 'block' AND g.is_fraud
  ORDER BY s.risk_score DESC, s.txn_id
  LIMIT 1""").collect()[0].asDict()
TRACED_TXN_ID = trace["txn_id"]
print("TRACED TXN_ID:", TRACED_TXN_ID)
for k, v in trace.items():
    if k != "top_shap_signals":
        print(f"  {k:22s} {v}")
print("  top_shap_signals:")
for s in json.loads(trace["top_shap_signals"]):
    print(f"    {s['feature']:24s} value={s['value']!s:>12}  shap={s['shap']:+.4f}")

cohort_n = spark.sql(f"""SELECT count(*) n, count_if(s.recommended_action='block') blk, avg(s.risk_score) avg_score
  FROM {SCORED_TABLE} s JOIN {FEATURE_TABLE} g USING (txn_id)
  WHERE g.amt_to_monthly_spend >= {COHORT_P90} AND g.is_cross_border = 1 AND g.is_new_device = 1""").collect()[0]
print(f"\nSeeded cohort across all rows: {cohort_n.n} txns, {cohort_n.blk} blocked, mean risk_score {cohort_n.avg_score:.4f}")